# `gnm` Interactive Tutorial: Dynamic Gauss-Newton Metropolis MCMC

This notebook introduces **`gnm`** ([Ugurbil & Goodman, arXiv:2001.03530](https://arxiv.org/abs/2001.03530)), a fast, strictly affine-invariant Markov chain Monte Carlo (MCMC) sampler for nonlinear inverse problems and stiff Bayesian posteriors of the form:

$$\pi(x) \propto \exp\left(-\frac{1}{2}\|f(x)\|_2^2 - \frac{1}{2}(x - m)^T H (x - m)\right)$$

We walk through four practical examples:
1. **1D Bimodal Double-Well Posterior** (comparing `gnm` histogram + error bars against exact quadrature)
2. **2D Stiff Rosenbrock Banana ($b = 100$)** with MAP warm-starting (`from_least_squares`) and dynamic cubic-Hermite back-off
3. **4D Ill-Posed Multi-Exponential Time Series Inversion** ($y(t) = w_1 e^{-\lambda_1 t} + w_2 e^{-\lambda_2 t}$)
4. **Black-Box Residual Models** (automatic finite-difference Jacobians) and **High-Dimensional Step-Size Tuning (`tune()`)**

In [ ]:
# Install gnm with Numba JIT acceleration and SciPy utilities (if running on Google Colab)
!pip install -q "gnm[fast,scipy]" matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
import gnm

print("gnm version:", gnm.__version__)

## 1. 1D Bimodal Double-Well Posterior

Consider the nonlinear residual $f(x) = (x^2 - y) / \sigma$ with $y = 2.0, \sigma = 1.0$, and a standard normal prior $x \sim \mathcal{N}(0, 1)$. Notice that $J(0) = 0$ at the barrier between the two modes $x \approx \pm\sqrt{2}$; `gnm` automatically regularizes the rank-deficient Jacobian at $x=0$ and transitions cleanly between both modes.

In [ ]:
def well_model(x, args):
    y, s = args["y"], args["s"]
    return 1, [(x[0] ** 2 - y) / s], [[(2.0 * x[0]) / s]]

np.random.seed(42)
sampler = gnm.sampler([0.5], well_model, {"y": 2.0, "s": 1.0})
sampler.prior([0.0], [[1.0]])

# Verify analytical Jacobian against finite differences (returns 0 on pass)
assert sampler.Jtest([-3.0], [3.0]) == 0

sampler.dynamic(2)
sampler.sample(20000)
sampler.burn(2000)

print(f"Acceptance rate:      {sampler.accept_rate:.3f}")
print(f"Autocorrelation time: {sampler.acor()[0]:.2f} steps")

# Plot sampled posterior vs. exact quadrature
x_grid, p_est, err = sampler.error_bars(50, [-3.0], [3.0])
x_fine = np.linspace(-3.0, 3.0, 400)
Z_norm, _ = integrate.quad(lambda u: sampler.posterior([u]), -6.0, 6.0)
p_exact = np.array([sampler.posterior([u]) / Z_norm for u in x_fine])

plt.figure(figsize=(7, 4))
plt.plot(x_fine, p_exact, "k-", lw=2, label="Exact Posterior (Quadrature)")
plt.errorbar(x_grid[0], p_est, yerr=err, fmt="o", color="#059669", markersize=4, capsize=2, label="GNM-Dynamic Samples")
plt.xlabel("$x$")
plt.ylabel("Posterior Density $\\pi(x)$")
plt.title("1D Bimodal Double-Well Posterior Recovery")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 2. 2D Stiff Rosenbrock Banana ($b = 100$) with MAP Warm-Starting

Using `gnm.sampler.from_least_squares`, we can automatically initialize the chain at the Maximum Likelihood / MAP point via `scipy.optimize.least_squares` and sample the narrow banana valley using dynamic cubic-Hermite back-off:

In [ ]:
def rosenbrock_2d(x, args):
    a, sb = args["a"], args["sb"]
    f = [a - x[0], sb * (x[1] - x[0] ** 2)]
    J = [[-1.0, 0.0], [-2.0 * sb * x[0], sb]]
    return 1, f, J

np.random.seed(7)
rosen_sampler = gnm.sampler.from_least_squares(
    rosenbrock_2d,
    x0=[-1.5, 2.5],
    args={"a": 1.0, "sb": 10.0},
)
print("Warm-started at MAP/MLE:", rosen_sampler._X["x"].ravel())

rosen_sampler.dynamic(2)
rosen_sampler.sample(15000)
chain_rosen = rosen_sampler.burn(1500)

plt.figure(figsize=(6.5, 4.2))
plt.scatter(chain_rosen[::5, 0], chain_rosen[::5, 1], s=4, alpha=0.35, color="#059669", label="GNM-Dynamic Samples")
plt.plot(1.0, 1.0, "r*", markersize=12, label="True Mode (1, 1)")
plt.xlabel("$x_0$")
plt.ylabel("$x_1$")
plt.title("2D Stiff Rosenbrock Banana ($b=100$) Samples")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 3. 4D Ill-Posed Multi-Exponential Time Series Inversion

Fitting noisy observations to a sum of decaying exponentials $y(t) = w_1 e^{-\lambda_1 t} + w_2 e^{-\lambda_2 t}$ is notoriously ill-conditioned because $(w_1, \lambda_1)$ and $(w_2, \lambda_2)$ are strongly collinear. `gnm`'s local Gauss-Newton metric $J(x)^T J(x) + H$ preconditions the collinear directions automatically:

In [ ]:
np.random.seed(3)
t_grid = np.linspace(0.0, 2.0, 15)
x_true = np.array([1.0, 2.0, 0.5, 2.5])  # [w1, w2, lambda1, lambda2]
y_clean = x_true[0] * np.exp(-x_true[2] * t_grid) + x_true[1] * np.exp(-x_true[3] * t_grid)
y_obs = y_clean + 0.05 * np.random.randn(t_grid.size)

def exp_ts_model(x, args):
    t, y, s = args["t"], args["y"], args["s"]
    e1 = np.exp(-x[2] * t)
    e2 = np.exp(-x[3] * t)
    f = (x[0] * e1 + x[1] * e2 - y) / s
    J = np.column_stack([e1 / s, e2 / s, -x[0] * t * e1 / s, -x[1] * t * e2 / s])
    return 1, f, J

ts_sampler = gnm.sampler(x_true, exp_ts_model, {"t": t_grid, "y": y_obs, "s": 0.1})
ts_sampler.prior(x_true, np.eye(4) * 0.5)
ts_sampler.dynamic(2)
ts_sampler.sample(10000)
ts_chain = ts_sampler.burn(1000)

print(f"Acceptance rate:        {ts_sampler.accept_rate:.3f}")
print(f"Autocorrelation times:  {np.round(ts_sampler.acor(), 2)}")
print(f"Posterior mean:         {np.round(np.mean(ts_chain, axis=0), 3)}")
print(f"Ground truth:           {x_true}")

## 4. Black-Box Residuals (Automatic Jacobians) & High-Dimensional `tune()`

If you only provide the residual vector $f(x)$, `gnm` automatically computes central finite-difference Jacobians. For higher-dimensional problems, `sampler.tune()` adapts the initial stage-0 step size $r_0 \in (0, 1]$ during warmup and freezes it for sampling:

In [ ]:
# Black-box residual function: no Jacobian needed!
def blackbox_residual(x):
    return [1.0 - x[0], 5.0 * (x[1] - x[0] ** 2), 1.0 - x[2], 5.0 * (x[3] - x[2] ** 2)]

bb_sampler = gnm.sampler([0.8, 0.8, 0.8, 0.8], blackbox_residual, r0="auto")
bb_sampler.dynamic(1)
tuned_r0 = bb_sampler.tune(n_tune=500, target_accept=0.45)
bb_sampler.sample(4000)
bb_chain = bb_sampler.burn(500)

print(f"Tuned r0:         {tuned_r0:.3f}")
print(f"Acceptance rate:  {bb_sampler.accept_rate:.3f}")
print(f"Posterior means:  {np.round(np.mean(bb_chain, axis=0), 3)}")